# ML Zoomcamp 2026 — Homework 6

Official: `materials/homework.md` (do not edit).
Data: `data/car_fuel_efficiency_2026.csv` (2026 pinned release).
Write-up: `ML_06_HW.md`.

Kernel: this folder `.venv`. Do not use `!pip` / `!wget`.
Submit: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw06


## 0) Imports


In [ ]:
# E: 캐시·임시·스레드 환경을 먼저 맞춘 뒤, 숙제에 쓸 라이브러리를 불러온다.

import os
os.environ.setdefault("MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
os.environ.setdefault("TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("TMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
# - 이름: os.environ.setdefault, numpy/pandas/xgboost,
#   DictVectorizer, root_mean_squared_error, train_test_split,
#   DecisionTreeRegressor, export_text, RandomForestRegressor
# - os.environ.setdefault 기능: 환경 변수가 없을 때만 기본값을 넣는다.
# - os.environ.setdefault 목적: Matplotlib·임시 파일을 C:가 아닌 E: 캐시에 두고,
#   수치 라이브러리 스레드를 1로 고정하기 위함.
# - DictVectorizer 기능: 행 딕셔너리를 숫자 행렬(원-핫 포함)로 바꾼다.
# - root_mean_squared_error 기능: 예측과 정답의 RMSE를 구한다.
# - DecisionTreeRegressor / RandomForestRegressor / xgb 기능: 회귀용 트리·숲·부스팅 모델.
# - export_text 기능: 학습된 나무를 들여쓰기 텍스트로 보여 준다.
# - 이 시점: 데이터 로드 전에 경로·도구를 준비하는 단계.


## 1) Load data and split

Fill missing values with zeros. Split with the exact calls from the homework.


In [ ]:
# 로컬 CSV를 읽고 결측을 0으로 채운 뒤, 숙제에 적힌 그대로 60/20/20으로 나눈다.
# DictVectorizer(sparse=True)로 훈련·검증 행렬을 만든다.

df = pd.read_csv("data/car_fuel_efficiency_2026.csv")
df = df.fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

del df_train["fuel_efficiency_mpg"]
del df_val["fuel_efficiency_mpg"]

train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

df.shape, X_train.shape, X_val.shape
# - 이름: read_csv, fillna, train_test_split, .values, del,
#   to_dict, DictVectorizer, fit_transform, transform, .shape
# - read_csv 기능: CSV를 DataFrame으로 읽는다.
# - fillna(0) 기능·목적: 결측을 0으로 채워 숙제 준비 규칙을 맞춘다.
# - train_test_split 기능: 표를 비율로 두 덩어리로 나눈다.
# - train_test_split 목적: 먼저 20% 테스트, 남은 80%를 75:25로 나눠
#   전체 기준 훈련 60% / 검증 20% / 테스트 20%를 만든다.
# - .values 기능: 연비 타깃 열을 NumPy 배열로 꺼낸다.
# - del ... fuel_efficiency_mpg 목적: 특성에 정답이 섞이지 않게 한다.
# - to_dict(orient="records") 기능: 행마다 {컬럼: 값} 리스트를 만든다.
# - fit_transform 기능: 범주 고유 값을 학습하며 훈련 행렬을 만든다.
# - transform 기능: 같은 컬럼 순서로 검증 행렬만 변환한다.
# - sparse=True 목적: 원-핫이 많은 표를 희소 행렬로 저장한다.
# - 이 시점: Q1부터 쓸 X_train / X_val / y_train / y_val 을 만드는 단계.


## Q1 — Decision tree, max_depth=1

Which original feature is used for splitting? Map a one-hot name such as `origin=USA` back to the original column.

- `vehicle_weight`
- `model_year`
- `origin`
- `fuel_type`

Answer:



In [ ]:
# 깊이 1짜리 회귀 나무를 학습해, 첫 분기(루트 분할)에 쓰인 특성을 텍스트로 확인한다.

dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=dv.feature_names_))
# - 이름: DecisionTreeRegressor, max_depth, .fit, export_text, feature_names_
# - DecisionTreeRegressor 기능: 수치 타깃용 결정 나무를 만든다.
# - max_depth=1 기능·목적: 질문을 한 번만 허용해 루트 분할 특성만 보게 한다.
# - .fit 기능: 훈련 행렬과 연비 타깃으로 분할 규칙을 학습한다.
# - export_text 기능: 학습된 규칙을 사람이 읽는 텍스트로 출력한다.
# - feature_names_ 기능: DictVectorizer가 만든 컬럼 이름 목록이다.
# - feature_names_ 목적: origin=USA 같은 원-핫 이름을 원본 열(origin)로
#   되짚어 선택지를 고르기 위함.
# - 이 시점: Q1 답을 고르기 위해 루트 분할을 확인하는 단계.


## Q2 — Random forest RMSE

`n_estimators=10`, `random_state=1`. Validation RMSE.

- 0.1837
- 1.837
- 18.37
- 183.7

Answer:



In [ ]:
# 나무 10그루 랜덤 포레스트를 학습하고 검증 셋 RMSE를 구해 선택지와 맞춘다.

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
root_mean_squared_error(y_val, y_pred)
# - 이름: RandomForestRegressor, n_estimators, random_state, n_jobs,
#   .fit, .predict, root_mean_squared_error
# - n_estimators=10 기능: 숲에 나무를 10그루 둔다.
# - random_state=1 기능·목적: 재현 가능한 난수로 숙제 숫자와 맞춘다.
# - n_jobs=-1 기능: 가능한 CPU를 써서 학습을 병렬로 돌린다(선택).
# - .predict 기능: 검증 행렬에 대한 연비 예측값을 낸다.
# - root_mean_squared_error 기능: 검증 정답과 예측의 RMSE를 계산한다.
# - 이 시점: Q2 선택지(소수 자릿수 스케일)를 고르는 단계.


## Q3 — n_estimators

Try `[10, 50, 100, 150]`, `random_state=1`. Which value has the lowest validation RMSE (3 decimal places)?

- 10
- 50
- 100
- 150

Answer:



In [ ]:
# n_estimators 후보마다 같은 검증 셋으로 RMSE를 재고, 소수 셋째 자리까지 반올림해 비교한다.

scores = []
for n in [10, 50, 100, 150]:
    model = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    model.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_val, model.predict(X_val))
    scores.append((n, round(rmse, 3)))
scores
# - 이름: for, RandomForestRegressor, .fit, .predict,
#   root_mean_squared_error, round, .append
# - for n in [...] 기능: 나무 개수 후보를 하나씩 돌린다.
# - round(rmse, 3) 기능·목적: 숙제가 말한 소수 셋째 자리 기준으로 비교한다.
# - scores 기능: (n_estimators, 반올림 RMSE) 쌍을 모은다.
# - 이 시점: Q3에서 검증 RMSE가 가장 낮은 n을 고르는 단계.


## Q4 — max_depth by mean RMSE

`max_depth` in `[10, 15, 20, 25]`. For each, `n_estimators` in `[10, 50, 100, 150]`. `random_state=1`.

- 10
- 15
- 20
- 25

Answer:



In [ ]:
# 깊이마다 나무 개수 네 값의 검증 RMSE 평균을 구해, 평균이 가장 좋은 max_depth를 고른다.

depth_scores = []
for depth in [10, 15, 20, 25]:
    rmses = []
    for n in [10, 50, 100, 150]:
        model = RandomForestRegressor(
            n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1
        )
        model.fit(X_train, y_train)
        rmses.append(root_mean_squared_error(y_val, model.predict(X_val)))
    depth_scores.append((depth, round(float(np.mean(rmses)), 3)))
depth_scores
# - 이름: 이중 for, max_depth, n_estimators, np.mean, round, float
# - 바깥 for depth 기능: 깊이 후보를 순회한다.
# - 안쪽 for n 기능: 같은 깊이에서 나무 개수를 바꿔 RMSE를 모은다.
# - np.mean(rmses) 기능: 그 깊이의 RMSE 네 값 평균을 낸다.
# - np.mean 목적: 숙제가 말한 mean RMSE로 깊이를 고르기 위함.
# - 이 시점: Q4에서 평균 검증 RMSE가 가장 좋은 max_depth를 고르는 단계.


## Q5 — Most important feature

`n_estimators=10`, `max_depth=20`, `random_state=1`. Among `vehicle_weight`, `horsepower`, `acceleration`, `engine_displacement`.

Answer:



In [ ]:
# 지정 파라미터로 숲을 학습한 뒤 feature_importances_를 이름과 짝지어 상위 특성을 본다.

rf = RandomForestRegressor(
    n_estimators=10, max_depth=20, random_state=1, n_jobs=-1
)
rf.fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=dv.feature_names_)
importances.sort_values(ascending=False).head(10)
# - 이름: feature_importances_, pd.Series, sort_values, head
# - feature_importances_ 기능: 분할 이득을 바탕으로 특성별 중요도 배열을 담는다.
# - pd.Series(..., index=feature_names_) 기능: 중요도에 컬럼 이름을 붙인다.
# - sort_values(ascending=False) 기능: 중요도가 큰 순으로 정렬한다.
# - head(10) 기능: 상위 10개만 보여 선택지 네 개와 대조하기 쉽게 한다.
# - 이 시점: Q5에서 가장 중요한 원본 특성을 고르는 단계.


## Q6 — XGBoost eta

100 rounds. Compare `eta` 0.3 and 0.1 on validation RMSE.

- 0.3
- 0.1
- Both give equal value

Answer:



In [ ]:
# 훈련·검증을 DMatrix로 감싼 뒤, eta 0.3과 0.1을 100라운드씩 학습해 검증 RMSE를 비교한다.

features = dv.feature_names_
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, "train"), (dval, "val")]

xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}

scores = {}
for eta in (0.3, 0.1):
    params = dict(xgb_params)
    params["eta"] = eta
    model = xgb.train(params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    y_pred = model.predict(dval)
    scores[eta] = root_mean_squared_error(y_val, y_pred)
scores
# - 이름: DMatrix, watchlist, xgb_params, eta, objective,
#   xgb.train, num_boost_round, evals, verbose_eval, .predict
# - DMatrix 기능: XGBoost용 행렬·라벨·특성 이름 컨테이너다.
# - watchlist 기능: 학습 중 평가할 (데이터, 이름) 목록이다.
# - eta 기능: 학습률(각 나무가 오차를 보정하는 보폭)이다.
# - objective="reg:squarederror" 기능·목적: 연비 회귀에 제곱오차 목표를 쓴다.
# - num_boost_round=100 기능: 나무를 100그루 순차로 쌓는다.
# - verbose_eval=False 기능: 라운드 로그를 찍지 않는다.
# - scores[eta] 목적: 두 학습률의 검증 RMSE를 나란히 비교한다.
# - 이 시점: Q6에서 더 좋은 eta(또는 동점)를 고르는 단계.
